In [16]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import pandas as pd
import matplotlib as mt 
from datasets import load_dataset
from transformers import AutoTokenizer

from torch.utils.data import DataLoader, Dataset

In [17]:
pd_dataset = pd.read_json('dataset/dataset.json')

In [18]:
pd_dataset = pd_dataset[pd_dataset['class'].apply(lambda x: x.get('personal_info') is not None)]
pd_dataset = pd_dataset[pd_dataset['class'].apply(lambda x: x.get('summary') is not None)]
pd_dataset = pd_dataset[pd_dataset['class'].apply(lambda x: x.get('skills') is not None)]
pd_dataset = pd_dataset[pd_dataset['class'].apply(lambda x: x.get('experience') is not None)]
pd_dataset = pd_dataset[pd_dataset['class'].apply(lambda x: x.get('education') is not None)]
pd_dataset = pd_dataset[pd_dataset['class'].apply(lambda x: x.get('certificates') is not None)]
pd_dataset = pd_dataset[pd_dataset['class'].apply(lambda x: x.get('objective') is not None)]

In [19]:
tokeniser = AutoTokenizer.from_pretrained('google-bert/bert-base-cased')

In [20]:
num_classes = 7

In [ ]:
class ResumeSectionsDataset(Dataset):
    def __init__(self, dataset, tokeniser):
        self.dataset = dataset
        self.tokeniser = tokeniser

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        ## feature
        raw_feature = self.dataset.iloc[idx]['feature']
        tokenised_data = self.tokeniser(text=raw_feature, truncation=True, padding="max_length", max_length=400, return_tensors="pt")
        feature = tokenised_data['input_ids'].squeeze(0).long()

        ## label
        label = torch.full((400,), -100, dtype=torch.long)

        personal_info_arr = self.dataset.iloc[idx]['class']['personal_info']
        summary_arr = self.dataset.iloc[idx]['class']['summary']
        skills_arr = self.dataset.iloc[idx]['class']['skills']
        experience_arr = self.dataset.iloc[idx]['class']['experience']
        education_arr = self.dataset.iloc[idx]['class']['education']
        certificates_arr = self.dataset.iloc[idx]['class']['certificates']
        objective_arr = self.dataset.iloc[idx]['class']['objective']

        label[personal_info_arr[0]: personal_info_arr[1]] = 0
        label[summary_arr[0]: summary_arr[1]] = 1
        label[skills_arr[0]: skills_arr[1]] = 2
        label[experience_arr[0]: experience_arr[1]] = 3
        label[education_arr[0]: education_arr[1]] = 4
        label[certificates_arr[0]: certificates_arr[1]] = 5
        label[objective_arr[0]: objective_arr[1]] = 6

        return label, feature

In [ ]:
pt_dataset = ResumeSectionsDataset(dataset=pd_dataset, tokeniser=tokeniser)

In [23]:
dataloader = DataLoader(dataset=pt_dataset, batch_size=60, shuffle=True)

In [ ]:
class ResumeSectionsModel(nn.Module):
    def __init__(self):
        super().__init__()

        self.embed = nn.Embedding(30000, 20)
        self.lstm = nn.LSTM(20, 20, batch_first=True)
        self.linear = nn.Linear(20, 7)

    def forward(self, feature):
        x = self.embed(feature)
        x, _ = self.lstm(x)
        logits = self.linear(x)

        return logits

In [ ]:
model = ResumeSectionsModel()

In [26]:
optimiser = optim.AdamW(model.parameters(), lr=0.001)

In [27]:
criterion = nn.CrossEntropyLoss(ignore_index=-100)

In [28]:
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer=optimiser, factor=0.9)

In [ ]:
for i in range(1000):
    loss = 0
    for batch in dataloader:
        label, feature = batch

        prediction = model(feature)

        loss = criterion(prediction.view(-1, num_classes), label.view(-1))

        print(f"Loss: {loss}")

        optimiser.zero_grad()
        loss.backward()
        optimiser.step()

    scheduler.step(loss.item())

Loss: 1.9726444482803345
torch.Size([60, 400, 7])
Loss: 1.9669617414474487
torch.Size([18, 400, 7])
Loss: 1.9652408361434937
torch.Size([60, 400, 7])
Loss: 1.9603463411331177
torch.Size([18, 400, 7])
Loss: 1.9575990438461304
torch.Size([60, 400, 7])
Loss: 1.9547988176345825
torch.Size([18, 400, 7])
Loss: 1.9505361318588257
torch.Size([60, 400, 7])
Loss: 1.9473166465759277
torch.Size([18, 400, 7])
Loss: 1.9439111948013306
torch.Size([60, 400, 7])
Loss: 1.9379706382751465
torch.Size([18, 400, 7])
Loss: 1.9354631900787354
torch.Size([60, 400, 7])
Loss: 1.9342073202133179
torch.Size([18, 400, 7])
Loss: 1.9284803867340088
torch.Size([60, 400, 7])
Loss: 1.9250694513320923
torch.Size([18, 400, 7])
Loss: 1.9200224876403809
torch.Size([60, 400, 7])
Loss: 1.9203542470932007
torch.Size([18, 400, 7])
Loss: 1.9122366905212402
torch.Size([60, 400, 7])
Loss: 1.912086844444275
torch.Size([18, 400, 7])
Loss: 1.906628966331482
torch.Size([60, 400, 7])
Loss: 1.895155668258667
torch.Size([18, 400, 7])
Los

In [30]:
torch.save(model, 'model/model.pth')